# Positional encodings (sinusoidal, learned, RoPE, ALiBi, relative) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Inject order into attention, which otherwise treats a sequence as a set
The cells compare no-position ambiguity, sinusoidal waves, learned position vectors, rotary angle preservation and ALiBi distance bias.

In [ ]:
X=np.array([[1.,0.],[0.,1.]]); perm=X[::-1]; same=np.sort(X,axis=0)-np.sort(perm,axis=0)
plt.figure(figsize=(5,2)); plt.imshow(np.r_[X,perm],cmap='Blues'); plt.title('Without position, content multiset is unchanged')
assert np.allclose(same,0)

In [ ]:
pos=np.arange(8); pe=np.c_[np.sin(pos/10**0), np.cos(pos/10**0)]
plt.figure(figsize=(6,3)); plt.plot(pos,pe[:,0],'-o',label='sin'); plt.plot(pos,pe[:,1],'-s',label='cos'); plt.legend(); plt.title('Sinusoidal position features')
assert pe.shape==(8,2) and abs(pe[0,1]-1)<1e-9

In [ ]:
learned=np.array([[0.1,0.0],[0.0,0.2],[0.2,0.1]]); tok=np.ones((3,2)); summed=tok+learned
plt.figure(figsize=(5,3)); plt.imshow(summed,cmap='Greens'); plt.colorbar(); plt.title('Learned positions are added to token vectors')
assert np.allclose(summed[1],[1,1.2])

In [ ]:
angle=np.pi/4; R=np.array([[np.cos(angle),-np.sin(angle)],[np.sin(angle),np.cos(angle)]]); q=np.array([1.,0.]); k=np.array([0.,1.]); dot=(R@q)@(R@k)
plt.figure(figsize=(4,4)); plt.quiver([0,0],[0,0],[q[0],(R@q)[0]],[q[1],(R@q)[1]],angles='xy',scale_units='xy',scale=1); plt.xlim(-1,1); plt.ylim(-.2,1.2); plt.title('RoPE rotates vectors by position')
assert abs(dot-(q@k))<1e-9

In [ ]:
T=6; slope=.5; bias=-slope*np.abs(np.subtract.outer(np.arange(T),np.arange(T)))
plt.figure(figsize=(4,3)); plt.imshow(bias,cmap='magma'); plt.colorbar(); plt.title('ALiBi penalizes distant positions')
assert bias[0,5]<bias[0,1]